# Embeddings: giving categories a geometry

> How 'user 84113' becomes something a model can reason about — and why the same trick underlies word vectors, recommendation, and every vector database you have heard of.

Read this chapter at `/learn/12-embeddings-and-tabular/`. Exported from `src/content/chapters/12-embeddings-and-tabular.mdx` — edit there, not here.


[Chapter 3](/learn/03-the-shape-of-problems/) left a loose end dangling, and I've
been looking forward to picking it up.

One-hot encoding turns *k* categories into *k* columns. Which is fine for three
districts, and absurd for fifty thousand product IDs.

Today we fix it — and the fix turns out to be one of the loveliest ideas in the
whole field.

## The problem, concretely

In [ ]:
import numpy as np, matplotlib.pyplot as plt

for n_categories in [3, 500, 50_000, 30_000_000]:
    hidden = 128
    print(f"{n_categories:>11,} categories -> one-hot -> dense({hidden}): "
          f"{n_categories * hidden:>15,} weights")

That last row is a catalogue of thirty million products, and it needs nearly four
billion parameters in the first layer alone. Before any learning happens.

But there's a second problem, and it's deeper than the cost. One-hot vectors are
all **equidistant**.

In [ ]:
words = ["cat", "kitten", "dog", "bulldozer"]
onehot = np.eye(len(words))

print("cosine similarity between one-hot vectors:")
for i, a in enumerate(words):
    print("  ", " ".join(f"{float(onehot[i] @ onehot[j]):.0f}" for j in range(len(words))), " ", a)

Read that grid. Under one-hot, *cat* is exactly as similar to *kitten* as it is to
*bulldozer*. Every pair is orthogonal. Every pair scores zero.

We've encoded the categories perfectly and thrown away the only interesting thing
about them, which is how they relate to one another.

## An embedding is a learned lookup table

Here's the idea. Give each category a short vector of **learned** numbers.
Instead of 50,000 columns of zeros, 32 columns of meaningful floats.

In [ ]:
rng = np.random.default_rng(0)
vocab, dim = 8, 4
table = rng.normal(0, 0.5, (vocab, dim))     # the embedding matrix

ids = np.array([3, 0, 7, 3])                 # a batch of category ids
print("lookup by index:\n", table[ids].round(2))
print("\nsame as one-hot @ table:", np.allclose(np.eye(vocab)[ids] @ table, table[ids]))

That last line is the whole trick, and it's worth pausing on:

**A one-hot vector times a matrix is just selecting a row of that matrix.**

Think about it — a vector of all zeros with a single 1 in position 3, multiplied
by a matrix, picks out row 3 and zeroes everything else. All those multiplications
by zero produce zero. Obviously.

So instead of materialising a 50,000-wide vector of zeros and doing a 50,000-wide
matrix multiply, you just... index. Same maths, a rounding error of the cost.

An embedding table is `Vec<[f32; D]>` indexed by an enum discriminant, and lookup
is `table[id as usize]`.

What makes it interesting is that the payload isn't written by you. It's a
*parameter*, so [backpropagation](/learn/09-backpropagation/) fills it in.

And the gradient is sparse in a beautifully simple way: only the rows actually
used in a batch receive a gradient. Everything else is untouched. That's exactly
what makes a thirty-million-row table trainable at all — each step only touches
the few hundred rows it saw.

In [ ]:
for n_categories in [500, 50_000, 30_000_000]:
    print(f"{n_categories:>11,} categories:  one-hot+dense(128) {n_categories * 128:>14,}"
          f"   embedding(dim 32) {n_categories * 32:>13,}")

## What the numbers become

Now — the compression is nice. The compression is not the remarkable part.

The remarkable part is that the learned coordinates turn out to be **meaningful**,
because training pushes categories that get used in similar ways to similar
positions. Let's watch it happen.

In [ ]:
# Twelve items, three latent groups. Nothing tells the model about the groups.
items = ["cat", "kitten", "dog", "puppy", "hamster",
         "guitar", "piano", "violin", "drums",
         "python", "rust", "haskell"]
groups = [0, 0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2]
n = len(items)

# A co-occurrence matrix: items in the same group appear together more often.
rng = np.random.default_rng(1)
co = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        co[i, j] = rng.poisson(28 if groups[i] == groups[j] else 2)
np.fill_diagonal(co, 0)
co = (co + co.T) / 2                      # co-occurrence is symmetric

# Factorise it: find a low-dimensional E with E @ E.T ~ log(1 + co).
target = np.log1p(co)
E = rng.normal(0, 0.1, (n, 2))
for step in range(4000):
    E -= 0.01 * 2 * (E @ E.T - target) @ E

print(f"reconstruction error: {np.abs(E @ E.T - target).mean():.3f}")

# Every raw embedding shares one big common component — the matrix is entirely
# non-negative, so every vector points roughly the same way. Centring removes it,
# and is standard practice on real embeddings for exactly this reason.
E = E - E.mean(axis=0)

In [ ]:
plt.figure(figsize=(5.4, 4))
colours = ["#1c6b58", "#9c4526", "#55467f"]
for i, (name, g) in enumerate(zip(items, groups)):
    plt.scatter(E[i, 0], E[i, 1], c=colours[g], s=40)
    plt.annotate(name, (E[i, 0], E[i, 1]), fontsize=8,
                 xytext=(4, 3), textcoords="offset points")
plt.title("2-D embedding, learned from co-occurrence alone")
plt.xticks([]); plt.yticks([]); plt.tight_layout()

Look at that picture. The animals are together. The instruments are together. The
programming languages are together.

Now — nobody told the model any of that. Nobody said *cat* and *kitten* are
related, or that *python* and *rust* belong in the same family. The model saw
only a matrix of which items co-occur, and it was asked to reproduce that matrix
from two numbers per item.

The geometry fell out because **that's what makes the reconstruction error
small**. Putting similar things near each other is simply the most efficient way
to compress "these things co-occur."

This is the whole idea, and it generalises much further than it looks:

**Meaning becomes distance.**

Once categories live in a vector space, "similar" stops being a lookup and
becomes a *computation*. And every downstream trick — recommendation, retrieval,
analogy, clustering, semantic search — is geometry from there on.

## Word vectors, and the famous analogy

The same procedure on text, at scale, was **word2vec** (2013) and **GloVe**
(2014). Train a model to predict a word from its neighbours, then throw the model
away and keep the embedding table. The table was always the point.

In [ ]:
def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

for a, b in [(0, 1), (0, 5), (0, 10), (9, 10), (5, 6)]:
    print(f"{items[a]:8s} ~ {items[b]:9s} {cosine(E[a], E[b]):+.3f}")

Cosine similarity is a dot product with the lengths
divided out — large when two vectors point the same way, negative when they point
apart. Within a group it's near +1; across groups it's negative.

That single operation is the engine of every semantic search system currently in
production. One dot product.

A note on the centring step in the previous cell, because it wasn't cosmetic and
it taught me something when I first hit it. A co-occurrence matrix is entirely
non-negative, so its single largest component is "how common is this item at
all" — a direction that *every* vector shares. That shared direction drags every
cosine toward +1 and drowns the actual signal.

I saw this the hard way while writing this chapter: before centring, *guitar* and
*rust* scored 0.996 similar. Subtracting the mean costs nothing and recovers the
structure completely. Real embedding pipelines do the same thing, sometimes under
the name "all-but-the-top".

Word2vec's celebrated result was that
`vec("king") − vec("man") + vec("woman")` lands near `vec("queen")`. The vector
space had apparently learned a "gender" direction and a "royalty" direction,
without anybody specifying either. It's a genuinely striking demonstration and it
deserved the attention it got.

Two caveats the popular retelling tends to leave out, though.

The result is real but **fragile**. It holds for a curated set of analogies and
often fails outside it, and the standard evaluation excludes the query words
themselves from the candidate answers — which quietly does a lot of the work.

And the same mechanism reproduces every bias in the training corpus, with exactly
the same confidence. `doctor − man + woman` returning `nurse` is not a bug in the
algorithm. It's an accurate summary of how those words were used in the text it
was shown.

Which is the clearest, most concrete example of a general truth worth carrying:
**a model learns the distribution it was shown, including the parts you wish
weren't in it.** It has no way to know which parts you'd have preferred.

## Embeddings for tabular data

Now back to the spreadsheet.

The technique that made neural networks genuinely competitive on tabular data —
pioneered for a Kaggle competition on retail sales forecasting, and later adopted
by fastai — is disarmingly simple: **one embedding table per categorical column,
concatenated with the continuous columns, into an ordinary dense network.**

In [ ]:
# needs PyTorch (this kernel has it; the browser runtime does not)
import torch, torch.nn as nn

class TabularNet(nn.Module):
    """Embeddings for categories, plain numbers for the rest, dense on top."""
    def __init__(self, cardinalities, n_continuous, hidden=64, n_out=1):
        super().__init__()
        # Rule of thumb (fastai): dim = min(50, (cardinality + 1) // 2)
        self.embeds = nn.ModuleList([
            nn.Embedding(card, min(50, (card + 1) // 2)) for card in cardinalities
        ])
        emb_total = sum(e.embedding_dim for e in self.embeds)
        self.body = nn.Sequential(
            nn.Linear(emb_total + n_continuous, hidden), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden, n_out),
        )

    def forward(self, x_cat, x_cont):
        parts = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeds)]
        return self.body(torch.cat([*parts, x_cont], dim=1))

model = TabularNet(cardinalities=[7, 12, 400], n_continuous=3)
print(model)
print("parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
# needs PyTorch (this kernel has it; the browser runtime does not)
torch.manual_seed(0)
batch = 16
x_cat  = torch.stack([torch.randint(0, c, (batch,)) for c in [7, 12, 400]], dim=1)
x_cont = torch.randn(batch, 3)
out = model(x_cat, x_cont)
print("cat input ", tuple(x_cat.shape), " cont input", tuple(x_cont.shape))
print("output    ", tuple(out.shape))
print("\nembedding dims:", [e.embedding_dim for e in model.embeds])

Day-of-week gets 4 dimensions. Month gets 6. Store-id gets 50.

And what the network can now learn is genuinely nicer than what one-hot allowed:
that certain stores behave alike, that December resembles November more than it
resembles June. Relationships that one-hot encoding makes *structurally
impossible* to express, because under one-hot every month is equally unlike every
other month.

This is genuinely useful, and it did **not** dethrone gradient boosting. I'd
rather you hear that from me than discover it in a retrospective.

On most tabular problems, `HistGradientBoostingRegressor` with sensible
categorical handling still matches or beats an embedding network — in a fraction
of the time, and with far fewer decisions available to get wrong.

Where embeddings genuinely win: very high-cardinality categories (millions of
users), when you want to **reuse** the learned representation somewhere else, and
when the tabular data sits alongside text or images in one model.

Those are real situations, and they're not the common case.
[Chapter 7](/learn/07-the-model-zoo/) still stands.

**"Where do the embedding *values* come from initially?"** Random, like any other
weight. They're meaningless at step zero and become meaningful through training —
which is the part that feels like magic and is just gradient descent.

**"How is `nn.Embedding` different from `nn.Linear`?"** It isn't, mathematically —
it's a Linear layer that takes indices instead of one-hot vectors, and skips the
multiplication by zero. That's the whole difference, and it's a large one at
scale.

**"What dimension should I pick?"** Try the rule of thumb, then try half and
double it, then look at validation. There's no theory here worth more than three
experiments — exercise 1 shows you why.

**"My embedding similarities are all near 1 and everything looks similar."**
That's exactly the shared-component problem described above. Centre your vectors
(subtract the mean) and try again. This bites people constantly with real
sentence embeddings.

**"`IndexError` from an embedding layer."** An id was ≥ the cardinality you
declared. Usually a category that appeared in validation but not training. Reserve
index 0 for "unknown" and map unseen categories to it.

## Where this goes: retrieval

Embeddings, plus cosine similarity, plus an index, is a
**vector database** — and that's most of what "semantic search" and the retrieval
half of RAG actually amount to.

In [ ]:
def nearest(query_idx, table, k=3):
    q = table[query_idx]
    sims = table @ q / (np.linalg.norm(table, axis=1) * np.linalg.norm(q) + 1e-12)
    order = np.argsort(sims)[::-1]
    return [(items[i], round(float(sims[i]), 3)) for i in order if i != query_idx][:k]

for q in [0, 6, 10]:
    print(f"{items[q]:9s} -> {nearest(q, E)}")

That is the entire algorithm. Six lines.

A production vector database adds an approximate nearest-neighbour index (HNSW,
IVF) so it doesn't have to compare against all thirty million rows, plus
persistence, plus metadata filtering. Real engineering, genuinely hard.

But the retrieval itself is a matrix multiply and an
argsort. When someone demos a vector database, that's what's
underneath.

Modern embeddings come from a transformer rather than a co-occurrence matrix, and
they embed whole sentences rather than single words.

The idea is completely unchanged: **learn a map from things to vectors, such that
useful similarity becomes geometric proximity.**

Everything after that is indexing.

In [ ]:
# 1. Re-run the co-occurrence factorisation with dim=1 instead of 2.
#    Can three groups be separated on a line? What does that tell you
#    about choosing embedding dimension?
#
# 2. Add a fourth group that overlaps with an existing one (share some
#    co-occurrence). Where does it land?
#
# 3. Normalise every row of E to unit length, then recompute the nearest
#    neighbours. Does the ranking change? Should it?

print("replace me")

For 1, get a piece of paper and try to place three dots on a *line* such that all
three pairs are equally far apart. Then think about what that means for the
model.

For 3, look carefully at what the `cosine` function already does to the vector
lengths before you run anything.

In [ ]:
def factorise(dim, steps=4000, lr=0.01, seed=1):
    r = np.random.default_rng(seed)
    Ed = r.normal(0, 0.1, (n, dim))
    for _ in range(steps):
        Ed -= lr * 2 * (Ed @ Ed.T - target) @ Ed
    return Ed, np.abs(Ed @ Ed.T - target).mean()

for dim in [1, 2, 3, 5, 10]:
    _, err = factorise(dim)
    print(f"dim {dim:2d}   reconstruction error {err:.4f}")

E1, _ = factorise(1)
E1 = E1 - E1.mean(0)
print("\n1-D positions:")
for name, g, v in sorted(zip(items, groups, E1.ravel()), key=lambda t: t[2]):
    print(f"  {v:+.2f}  {name:9s} (group {g})")

**Question 1** is the point of the whole exercise, and I hope the paper helped.

One dimension can *order* the items, but it cannot express "three mutually distant
clusters." A line has only two directions of separation, so one group is always
stuck in the middle, forced to be near both of the others. Look at the printed
positions — one group is sandwiched, and there was never anything the model could
have done about it.

Two dimensions suffice for three clusters. More dimensions keep helping, until
they don't.

And that is the entire story of choosing an embedding dimension: too small and
genuinely distinct things are forced to collide; too large and you're spending
parameters and inviting overfitting for no gain.

The fastai rule of thumb `min(50, (cardinality + 1) // 2)` isn't theory — it's a
shape that has worked across a lot of datasets. The honest method is still to try
three values and look at your validation score. That's not a failure of the
field; it's what empirical means.

In [ ]:
# 3. Normalising (E is already centred)
En = E / np.linalg.norm(E, axis=1, keepdims=True)
print("raw       :", nearest(0, E))
print("normalised:", nearest(0, En))

The ranking is unchanged — and it *must* be. Cosine similarity already divides
out the magnitudes, so normalising beforehand changes precisely nothing. If you
predicted that from reading the function, well done.

But it matters for a different and rather practical reason. If you
**pre-normalise** your table, then cosine similarity collapses into a plain dot
product — because the denominator is now always 1. And a plain dot product is a
single matrix multiply, which is enormously faster at thirty million rows.

That's why real vector databases store normalised vectors, and why their
similarity metric is usually called "inner product" rather than "cosine". Same
answer, less arithmetic. It's a nice example of how a mathematical identity turns
directly into an engineering decision.

Tomorrow: attention — how a model decides which parts of its input to look at, and
the architecture that ate the entire field.